<a href="https://colab.research.google.com/github/MohammadYehya/Sentiment_Analysis/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installing Dependencies

In [ ]:
# 1. Force upgrade the core libraries
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade transformers datasets accelerate

# 2. Fix the specific peft/transformers dependency loop
!pip install --upgrade peft

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
import pandas as pd
from datasets import load_dataset

# 1. Load Dataset
raw_datasets = load_dataset("imdb")

# 2. Convert to Pandas for Phase 1 & 4
train_df = pd.DataFrame(raw_datasets['train']).sample(2000, random_state=42) # Sampling for speed
test_df = pd.DataFrame(raw_datasets['test']).sample(500, random_state=42)

# Standardize labels
X_train, y_train = train_df['text'], train_df['label']
X_test, y_test = test_df['text'], test_df['label']

print(f"Data Loaded: {len(train_df)} training rows, {len(test_df)} testing rows.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Data Loaded: 2000 training rows, 500 testing rows.


## Logistic Regression + TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Vectorize
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train & Predict
lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)

print("--- Phase 1: Logistic Regression Results ---")
print(classification_report(y_test, lr_preds))

--- Phase 1: Logistic Regression Results ---
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       266
           1       0.83      0.83      0.83       234

    accuracy                           0.84       500
   macro avg       0.84      0.84      0.84       500
weighted avg       0.84      0.84      0.84       500



## Fine-Tuned DistilBERT

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

model_nm = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_nm)

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Prepare HuggingFace format
tokenized_ds = raw_datasets.map(tokenize_fn, batched=True)
small_train = tokenized_ds["train"].shuffle(seed=42).select(range(1000))
small_test = tokenized_ds["test"].shuffle(seed=42).select(range(500))

# Load Model
bert_model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)

# Training Specs
args = TrainingArguments(
    output_dir="bert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1, # Keep it low for the notebook demo
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(model=bert_model, args=args, train_dataset=small_train, eval_dataset=small_test)
trainer.train()

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,0.609023


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=0.6513886224655878, metrics={'train_runtime': 18.5639, 'train_samples_per_second': 53.868, 'train_steps_per_second': 3.394, 'total_flos': 33116849664000.0, 'train_loss': 0.6513886224655878, 'epoch': 1.0})

## Zero-Shot LLM

In [ ]:
from transformers import pipeline

zero_shot_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

def predict_zeroshot(text):
    labels = ["positive", "negative"]
    res = zero_shot_pipe(text, labels)
    return res['labels'][0], res['scores'][0]

sample_text = "The plot was nonsensical, but the acting was world-class."
label, score = predict_zeroshot(sample_text)
print(f"Zero-Shot Result: {label} ({score:.2f} confidence)")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Zero-Shot Result: positive (0.68 confidence)


## Custom PyTorch Transformer

In [ ]:
from collections import Counter
import torch

# 1. Build Vocabulary from IMDb train_df
# We use the top 10,000 words to keep the model efficient
all_text = " ".join(train_df['text'].values).lower().split()
vocab_size = 10000
common_words = Counter(all_text).most_common(vocab_size - 2)

# Map words to unique IDs (0 = Padding, 1 = Unknown)
word_to_idx = {word: i + 2 for i, (word, _) in enumerate(common_words)}
word_to_idx["<PAD>"] = 0
word_to_idx["<UNK>"] = 1

# 2. The critical helper function
def tweet_to_tensor(text, max_len=256):
    """Converts a string into a numerical tensor of length max_len."""
    tokens = text.lower().split()
    ids = [word_to_idx.get(t, 1) for t in tokens][:max_len]
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return torch.tensor([ids])

print("✅ Foundation ready: Vocabulary and tweet_to_tensor defined.")

✅ Foundation ready: Vocabulary and tweet_to_tensor defined.


In [ ]:
import torch.nn as nn
import math

class CustomTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=8, num_layers=3, max_len=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Positional Encoding Buffer (Non-learnable)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

        # Transformer Encoder Blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=512
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Final Classification Head
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, x):
        # Add word embeddings and position information
        x = self.embedding(x) + self.pe[:, :x.size(1), :]
        # Process through Transformer
        x = self.transformer(x)
        # Average the output vectors and classify
        return self.classifier(x.mean(dim=1))

pt_model = CustomTransformer(vocab_size=vocab_size)
print("✅ Architecture ready: pt_model initialized.")

✅ Architecture ready: pt_model initialized.


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pt_model.to(device)

# 1. Prepare Data
train_tensors = torch.cat([tweet_to_tensor(t, max_len=256) for t in train_df['text']])
train_labels = torch.tensor(train_df['label'].values, dtype=torch.float32).unsqueeze(1)
train_loader = DataLoader(TensorDataset(train_tensors, train_labels), batch_size=32, shuffle=True)

# 2. Train
optimizer = optim.AdamW(pt_model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

pt_model.train()
print("🚀 Starting Training...")
for epoch in range(3):
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(pt_model(texts), labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} complete.")

# 3. Final Inference function
def get_custom_pred(text):
    pt_model.eval()
    tensor = tweet_to_tensor(text, max_len=256).to(device)
    with torch.no_grad():
        logit = pt_model(tensor)
        prob = torch.sigmoid(logit).item()
    return "Positive" if prob > 0.5 else "Negative", prob

print("\n🏆 Tournament Ready! You can now call get_custom_pred(text).")

🚀 Starting Training...
Epoch 1 complete.
Epoch 2 complete.
Epoch 3 complete.

🏆 Tournament Ready! You can now call get_custom_pred(text).


## Tournament Inferrence

In [ ]:
def run_tournament(text):
    print(f"\n{'='*60}")
    print(f"TESTING TEXT: \"{text[:100]}...\"")
    print(f"{'='*60}")

    # --- 1. Traditional ML (Logistic Regression) ---
    # Needs TF-IDF vectorization
    tfidf_input = tfidf.transform([text])
    lr_pred_numeric = lr_model.predict(tfidf_input)[0]
    lr_label = "Positive" if lr_pred_numeric == 1 else "Negative"

    # --- 2. Fine-Tuned DistilBERT (Hugging Face) ---
    # Needs the BERT Tokenizer and GPU/CPU device management
    bert_inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    bert_model.eval()
    with torch.no_grad():
        bert_logits = bert_model(**bert_inputs).logits
        bert_pred_numeric = torch.argmax(bert_logits, dim=1).item()
    bert_label = "Positive" if bert_pred_numeric == 1 else "Negative"

    # --- 3. Custom Transformer (PyTorch Scratch) ---
    # Needs our custom 'tweet_to_tensor' function and vocabulary
    pt_input = tweet_to_tensor(text, max_len=256).to(device)
    pt_model.eval()
    with torch.no_grad():
        pt_logit = pt_model(pt_input)
        pt_prob = torch.sigmoid(pt_logit).item()
    pt_label = "Positive" if pt_prob > 0.5 else "Negative"

    # --- 4. Zero-Shot Classification (LLM) ---
    # Handles everything internally via the pipeline
    zs_result = predict_zeroshot(text)
    zs_label = zs_result[0].capitalize()
    zs_conf = zs_result[1]

    # --- FINAL COMPARISON TABLE ---
    print(f"{'MODEL TYPE':<25} | {'PREDICTION':<12} | {'CONFIDENCE/SCORE'}")
    print(f"{'-'*25}-|-{'-'*12}-|-{'-'*18}")
    print(f"{'1. Logistic (TF-IDF)':<25} | {lr_label:<12} | N/A")
    print(f"{'2. DistilBERT (Fine-tuned)':<25} | {bert_label:<12} | High")
    print(f"{'3. Custom Transformer':<25} | {pt_label:<12} | {pt_prob:.2%}")
    print(f"{'4. Zero-Shot (BART)':<25} | {zs_label:<12} | {zs_conf:.2%}")
    print(f"{'='*60}\n")

# --- TEST RUNS ---
# Test 1: Clear Positive
run_tournament("This was an incredible cinematic experience. I loved every second of it!")

# Test 2: Sarcasm/Nuance (The ultimate test for sentiment models)
run_tournament("The plot was as thin as a piece of paper and the acting was wooden, but hey, the popcorn was good.")

# Test 3: Mixed Sentiment
run_tournament("I expected to hate this movie, but it turned out to be surprisingly decent.")


TESTING TEXT: "This was an incredible cinematic experience. I loved every second of it!..."
MODEL TYPE                | PREDICTION   | CONFIDENCE/SCORE
--------------------------|--------------|-------------------
1. Logistic (TF-IDF)      | Positive     | N/A
2. DistilBERT (Fine-tuned) | Positive     | High
3. Custom Transformer     | Negative     | 49.64%
4. Zero-Shot (BART)       | Positive     | 99.49%


TESTING TEXT: "The plot was as thin as a piece of paper and the acting was wooden, but hey, the popcorn was good...."
MODEL TYPE                | PREDICTION   | CONFIDENCE/SCORE
--------------------------|--------------|-------------------
1. Logistic (TF-IDF)      | Negative     | N/A
2. DistilBERT (Fine-tuned) | Negative     | High
3. Custom Transformer     | Negative     | 48.56%
4. Zero-Shot (BART)       | Negative     | 95.96%


TESTING TEXT: "I expected to hate this movie, but it turned out to be surprisingly decent...."
MODEL TYPE                | PREDICTION   | CONFIDENCE/